In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables (.env file should contain GOOGLE_API_KEY)
load_dotenv()

# Initialize Gemini Chat Model
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", 
    temperature=0.1
)

print("✅ Gemini Chat Model initialized successfully!")

✅ Gemini Chat Model initialized successfully!


In [2]:
!pip install langchain-chroma chromadb
!pip install langchain-huggingface sentence-transformers

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

# 1. Define Examples
examples = [
    {"input": "کوئرا دقیقا چه کاری انجام میده؟", "output": "کوئرا جامعه برنامه‌نویسان ایران است که محصولاتی مثل مسابقات، بوت‌کمپ و کوئرا کالج دارد."},
    {"input": "می‌خوام برنامه نویسی تمرین کنم، سوال رایگان دارید؟", "output": "بله، در «بانک سوالات کوئرا» تعداد زیادی سوال رایگان در شاخه‌های الگوریتم، مهندسی نرم‌افزار و هوش مصنوعی وجود دارد."},
    {"input": "چه دوره‌های آموزشی دارید؟", "output": "در «کوئرا کالج» دوره‌های مختلف برنامه‌نویسی برای علایق مختلف وجود دارد که می‌توانید آموزش‌های جذابی در آن پیدا کنید."},
    {"input": "اگه زمان یادگیری و گذروندن دوره به مشکل خوردم یا درباره مطالب سوال داشتم از کی بپرسم؟", "output": "در کوئرا کالج، مربی‌های حرفه‌ای دوره و ربات کای در کنار شما هستند. هر زمان که نیاز به راهنمایی داشتید، برای پاسخ سریع و لحظه‌ای به کای مراجعه کنید یا برای پاسخ دقیق‌تر، از بخش سوال خصوصی، با مربی‌های دوره در ارتباط باشید."}
]

# 2. Initialize HuggingFace Embeddings (Local & Free)
# این مدل برای اولین بار حدود 80 مگابایت دانلود می‌شود و سپس کاملاً آفلاین کار می‌کند
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Create the Example Selector using Chroma DB
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples=examples,
    embeddings=embeddings,
    vectorstore_cls=Chroma,
    k=1, # فقط یک مثال که بیشترین شباهت را دارد انتخاب کن
)

print("✅ Example Selector and Vector Database created successfully with HuggingFace!")

c:\Users\aline\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4273.85it/s]


✅ Example Selector and Vector Database created successfully with HuggingFace!


In [4]:
# Test the semantic search
selected_example = example_selector.select_examples({"input": "مربی"})

print("--- Selected Example Based on Semantic Similarity ---")
print(selected_example)

--- Selected Example Based on Semantic Similarity ---
[{'input': 'می\u200cخوام برنامه نویسی تمرین کنم، سوال رایگان دارید؟', 'output': 'بله، در «بانک سوالات کوئرا» تعداد زیادی سوال رایگان در شاخه\u200cهای الگوریتم، مهندسی نرم\u200cافزار و هوش مصنوعی وجود دارد.'}]


In [5]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# بارگذاری متغیرهای محیطی برای خواندن API Key
load_dotenv()

# مقداردهی مجدد مدل گفت‌وگوی جمینای
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest", 
    temperature=0.1
)

from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. Define the template for exactly how each example should be formatted
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# 2. Create the dynamic Few-Shot prompt
dynamic_few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    input_variables=["input"],
)

# 3. Build the final prompt combining system message, dynamic examples, and user input
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "تو دستیار پشتیبانی شرکت کوئرا هستی. خیلی کوتاه و دقیق پاسخ بده."),
    dynamic_few_shot_prompt,
    ("human", "{input}"),
])

# 4. Create the pipeline (Chain)
chain = final_prompt | chat_model

print("✅ Dynamic Few-Shot Chain is ready!")

✅ Dynamic Few-Shot Chain is ready!


In [8]:
question = "اگه در حین گذروندن دوره، درباره مطلبی سوال داشتم، چه کاری کنم؟"

print("⏳ Processing dynamic retrieval and generating response...\n")
response = chain.invoke({"input": question})

print("--- Final Model Response ---")
print(response.content)

⏳ Processing dynamic retrieval and generating response...

--- Final Model Response ---
[{'type': 'text', 'text': 'می\u200cتوانید سوال خود را در انجمن پرسش و پاسخ دوره مطرح کنید تا مربیان راهنمایی\u200cتان کنند.', 'extras': {'signature': 'EmAKXgFpFH0Tc1oKmByGs9Q6ynZxUweztqEx7MCltZPl8FhcK0Tf9dlsk/1qjkOc/akdTdyWpc7wOMHs9YcMVPy1JaVKKggdPMEsjArVkjz2PC2MKCCnBI5UslP0QksB/Eg='}}]
